In [5]:
%matplotlib ipympl
from CrocoDash.grid_creator import GridCreator
gridc = GridCreator()
gridc

GridCreator(children=(VBox(children=(VBox(children=(HTML(value='<h3>Grid Creator &mdash; Create Mode</h3>'), R…

In [6]:
from CrocoDash.topo import Topo

topo = Topo(
    grid = gridc.grid,
    min_depth = 9.5, # in meters
)

In [7]:
from pathlib import Path

bathymetry_path= Path("/glade/derecho/scratch/manishrv/ice/croc/gebco/GEBCO_2024.nc")

topo.set_from_dataset(
    bathymetry_path = bathymetry_path,
    longitude_coordinate_name="lon",
    latitude_coordinate_name="lat",
    vertical_coordinate_name="elevation"
)

**NOTE**
            If bathymetry setup fails (e.g. kernel crashes), restart the kernel and edit this cell.
            Call ``[topo_object_name].mpi_set_from_dataset()`` instead. Follow the given instructions for using mpi
            and ESMF_Regrid outside of a python environment. This breaks up the process, so be sure to call
            ``[topo_object_name].tidy_dataset() after regridding with mpi.
Slicing source bathymetry to domain: (244.61899794103203, 247.76899794103204) x (29.565888562634107, 31.945888562634106) with buffer 0.5
Begin regridding dataset...

Original dataset size: 1.63 Mb
Regridded size: 0.02 Mb

Generating regridding weights using xESMF with method 'bilinear'...


/glade/work/manishrv/conda-envs/CrocoDash/lib/python3.11/site-packages/xesmf/backend.py:42: UserWarning: Input array is not F_CONTIGUOUS. Will affect performance.
  warnings.warn('Input array is not F_CONTIGUOUS. ' 'Will affect performance.')


Tidy bathymetry: Reading in regridded bathymetry to fix up metadata...done. Filling in inland lakes and channels... 

In [8]:
from CrocoDash.topo_editor import TopoEditor
TopoEditor(topo)

TopoEditor(children=(VBox(children=(HTML(value='<h2>Topo Editor</h2>'), VBox(children=(VBox(children=(HTML(val…

In [9]:
from CrocoDash.vgrid import VGrid

vgrid  = VGrid.hyperbolic(
    nk = 75, # number of vertical levels
    depth = topo.max_depth,
    ratio=20.0 # target ratio of top to bottom layer thicknesses
)

In [10]:
from CrocoDash.vgrid_creator import VGridCreator
vc = VGridCreator(vgrid)
vc

VGridCreator(children=(VBox(children=(VBox(children=(HTML(value='<h3>Vertical Grid Creator</h3>'), ToggleButto…

In [11]:
# CESM case (experiment) name
casename = "demo-case-1"

# CESM source root (Update this path accordingly!!!)
cesmroot ="/glade/u/home/manishrv/work/installs/CROCESM_workshop_2025"

# Place where all your input files go 
inputdir = Path.home()/"scratch" / "croc_input" / casename
    
# CESM case directory
caseroot = Path.home() / "croc_cases" / casename

In [12]:
from CrocoDash.case import Case

case = Case(
    cesmroot = cesmroot,
    caseroot = caseroot,
    inputdir = inputdir,
    ocn_grid = gridc.grid,
    ocn_vgrid = vc.vgrid,
    ocn_topo = topo,
    project = 'NCGD0011',
    override = True,
    machine = "derecho",
    compset = "CR_JRA" 
)

INFO:	initialize:Initializing the visualCaseGen system...
INFO:  csp_solver:CspSolver initialized.


Creating case...

• Updating ccs_config/modelgrid_aliases_nuopc.xml file to include the new resolution "demo-case-1" consisting of the following component grids.
 atm grid: "TL319" 
 lnd grid: "TL319" 
 ocn grid: "GofC2".
 rof grid: "null".

• Updating ccs_config/component_grids_nuopc.xml file to include newly generated ocean grid "GofC2" with the following properties:
 nx: 28, ny: 21. ocean mesh: /glade/u/home/manishrv/scratch/croc_input/demo-case-1/ocnice/ESMF_mesh_GofC2_42af53.nc.

Running the create_newcase tool with the following command:

/glade/u/home/manishrv/work/installs/CROCESM_workshop_2025/cime/scripts/create_newcase --compset CR_JRA --res demo-case-1 --case /glade/u/home/manishrv/croc_cases/demo-case-1 --machine derecho --run-unsupported --project NCGD0011 --non-local 

The create_newcase command was successful.

Navigating to the case directory:

cd /glade/u/home/manishrv/croc_cases/demo-case-1

Running the case.setup script with the following command:

./case.setup --no

INFO:	stage:SUCCESS: All stages are complete.


Case created successfully at /glade/u/home/manishrv/croc_cases/demo-case-1.

To further customize, build, and run the case, navigate to the case directory in your terminal. To create another case, restart the notebook.

./xmlchange MOM6_MEMORY_MODE=dynamic_symmetric --non-local



In [13]:
case.configure_forcings(
    date_range = ["2020-01-01 00:00:00", "2020-01-05 00:00:00"],
    function_name="get_glorys_data_from_cds_api"
)

INFO:CrocoDash.forcing_configurations.base:[SKIP] tides missing args: ['tpxo_elevation_filepath', 'tpxo_velocity_filepath', 'tidal_constituents']
INFO:CrocoDash.forcing_configurations.base:[SKIP] BGC incompatible with compset
INFO:CrocoDash.forcing_configurations.base:[SKIP] CICE incompatible with compset
INFO:CrocoDash.forcing_configurations.base:[SKIP] BGCIC incompatible with compset
INFO:CrocoDash.forcing_configurations.base:[SKIP] BGCIronForcing incompatible with compset
INFO:CrocoDash.forcing_configurations.base:[SKIP] BGCRiverNutrients incompatible with compset
INFO:CrocoDash.forcing_configurations.base:[SKIP] Runoff incompatible with compset
INFO:CrocoDash.forcing_configurations.base:[SKIP] Chl missing args: ['chl_processed_filepath']


Adding parameter changes to user_nl_mom:

  ! Initial conditions
  INIT_LAYERS_FROM_Z_FILE = True
  Z_INIT_ALE_REMAPPING = True
  TEMP_SALT_INIT_VERTICAL_REMAP_ONLY = True
  DEPRESS_INITIAL_SURFACE = True
  VELOCITY_CONFIG = file
  TEMP_SALT_Z_INIT_FILE = init_tracers.nc
  SURFACE_HEIGHT_IC_FILE = init_eta.nc
  VELOCITY_FILE = init_vel.nc
  Z_INIT_FILE_PTEMP_VAR = temp
  Z_INIT_FILE_SALT_VAR = salt
  SURFACE_HEIGHT_IC_VAR = eta_t
  U_IC_VAR = u
  V_IC_VAR = v

  ! Open boundary conditions
  OBC_NUMBER_OF_SEGMENTS = 4
  OBC_FREESLIP_VORTICITY = False
  OBC_FREESLIP_STRAIN = False
  OBC_COMPUTED_VORTICITY = True
  OBC_COMPUTED_STRAIN = True
  OBC_ZERO_BIHARMONIC = True
  OBC_TRACER_RESERVOIR_LENGTH_SCALE_OUT = 3.0E+04
  OBC_TRACER_RESERVOIR_LENGTH_SCALE_IN = 3000.0
  BRUSHCUTTER_MODE = True
  OBC_SEGMENT_001 = "J=0,I=0:N,FLATHER,ORLANSKI,NUDGED,ORLANSKI_TAN,NUDGED_TAN"
  OBC_SEGMENT_001_VELOCITY_NUDGING_TIMESCALES = 0.3, 360.0
  OBC_SEGMENT_001_DATA = "U=file:forcing_obc_segment_001.nc(u

In [14]:
case.process_forcings()

INFO - 2026-06-14T19:09:00Z - Selected dataset version: "202311"
INFO:copernicusmarine:Selected dataset version: "202311"
INFO - 2026-06-14T19:09:00Z - Selected dataset part: "default"
INFO:copernicusmarine:Selected dataset part: "default"
INFO - 2026-06-14T19:09:23Z - Starting download. Please wait...
INFO:copernicusmarine:Starting download. Please wait...


  0%|          | 0/7239 [00:00<?, ?it/s]

INFO - 2026-06-14T19:10:31Z - Successfully downloaded to /glade/derecho/scratch/manishrv/tmp/tmpwd38_ipn/test_file.notreal.nc
INFO:copernicusmarine:Successfully downloaded to /glade/derecho/scratch/manishrv/tmp/tmpwd38_ipn/test_file.notreal.nc
INFO:CrocoDash.extract_forcings.get_dataset_piecewise:Downloading GLORYS data using get_glorys_data_from_cds_api from 2020-01-01 00:00:00 to 2020-01-05 00:00:00.
INFO:CrocoDash.extract_forcings.get_dataset_piecewise:Using step size 5, this will result in 1 files per boundary.
INFO - 2026-06-14T19:10:34Z - Selected dataset version: "202311"
INFO:copernicusmarine:Selected dataset version: "202311"
INFO - 2026-06-14T19:10:34Z - Selected dataset part: "default"
INFO:copernicusmarine:Selected dataset part: "default"
INFO - 2026-06-14T19:10:53Z - Starting download. Please wait...
INFO:copernicusmarine:Starting download. Please wait...


  0%|          | 0/21511 [00:00<?, ?it/s]

INFO - 2026-06-14T19:13:13Z - Successfully downloaded to /glade/u/home/manishrv/scratch/croc_input/demo-case-1/extract_forcings/raw_data/ic_unprocessed.nc
INFO:copernicusmarine:Successfully downloaded to /glade/u/home/manishrv/scratch/croc_input/demo-case-1/extract_forcings/raw_data/ic_unprocessed.nc
INFO - 2026-06-14T19:13:16Z - Selected dataset version: "202311"
INFO:copernicusmarine:Selected dataset version: "202311"
INFO - 2026-06-14T19:13:16Z - Selected dataset part: "default"
INFO:copernicusmarine:Selected dataset part: "default"
INFO - 2026-06-14T19:13:32Z - Starting download. Please wait...
INFO:copernicusmarine:Starting download. Please wait...


...

No arakawa_grid provided, assuming the variable mapping for your data product is already in correct format.
Arakawa A grid detected in variable mapping


INFO:regional_mom6.rotation:Getting rotation angle
INFO:regional_mom6.rotation:Calculating grid rotation angle


Setting up Initial Conditions
Regridding Velocities... Done.
Regridding Tracers... Done.
Regridding Free surface... Done.
Saving outputs... 

INFO:CrocoDash.extract_forcings.regrid_dataset_piecewise:Start mom6_forge fill...
INFO:CrocoDash.extract_forcings.regrid_dataset_piecewise:...end mom6_forge fill.
INFO:CrocoDash.extract_forcings.regrid_dataset_piecewise:Finished regridding
INFO:CrocoDash.extract_forcings.merge_piecewise_dataset:Parsing Regridded Data Folder
INFO:CrocoDash.extract_forcings.merge_piecewise_dataset:All boundaries continuous and non-overlapping.
INFO:CrocoDash.extract_forcings.merge_piecewise_dataset:Merging Files


: 